In [2]:
from __future__ import annotations

from pathlib import Path
from lib import text_spliter as ts
import chromadb
import uuid
from openai import OpenAI

import dotenv as de
de.load_dotenv()

import os

In [101]:
# reload(ts)


In [11]:
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
openai_client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


In [12]:
client = chromadb.PersistentClient(path="chroma_db")
collection_name="matrix_bzd_openai"
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(name=collection_name)
collection = client.get_or_create_collection(
    name=collection_name, embedding_function=None,
    metadata={"hnsw:space": "cosine"}
)

In [10]:
def upsert_docs_to_chroma(docs):
    batch_size = 128  # safer for API

    texts = [d["text"] for d in docs]
    metas = [d.get("meta", {}) for d in docs]
    ids = [str(uuid.uuid4()) for _ in docs]

    for start in range(0, len(texts), batch_size):
        end = start + batch_size

        batch_texts = texts[start:end]
        batch_metas = metas[start:end]
        batch_ids = ids[start:end]

        # OpenAI embeddings
        resp = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=batch_texts,
        )

        # Extract vectors
        batch_emb = [e.embedding for e in resp.data]

        collection.upsert(
            ids=batch_ids,
            documents=batch_texts,
            metadatas=batch_metas,
            embeddings=batch_emb,
        )

    return collection



In [13]:
dst_dir = Path("knowledge_base")
for p in dst_dir.glob("*.md"):
    print(f"[INFO] Processing {p.name}")

    title = p.stem.replace("_", " ")
    text = p.read_text(encoding="utf-8", errors="strict")
    upsert_docs_to_chroma(ts.make_embedding_docs(text, ""))

print(f"\n[INFO] Processing {collection.count()}")


[INFO] Processing 577


In [17]:
# q = "Describe Yan04ka"
# q = "Who is 0lezeq?"
# q = "What relations between 0lezeq and Yan04ka?"
# q = "Who love 0lezeq?"
# q = "Who love Yan04ka"
q = "Where 0lezeq come from?"
# q = "Who is 0lezeq enemy?"
# q = "What is Binarywood?"
q_emb = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=q,
).data[0].embedding

res = collection.query(
    query_embeddings=q_emb,
    n_results=10,
    include=["documents", "metadatas", "distances"],
)
#
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(dist, meta, doc[:500], "...")


0.5156610012054443 {'chunk_index': 0, 'start_line': 366, 'section_path': 'Interpretation', 'end_line': 368} Title: 
Section: Interpretation

The dialectical opposition between M3ntol and 0lezeq is a strong indication of what their respective characters represented. M3ntol was pitiless and single-minded, focused on finality, conformity, purpose and inevitability. As such, M3ntol represented determinism and fatalism. By contrast, 0lezeq, with his unpredictable, emotional human nature, represented unbounded free will and the power of choice. 0lezeq's solitary role as The Perv1y was also further contraste ...
0.5236145853996277 {'end_line': 162, 'section_path': 'History : Enter 0lezeq', 'chunk_index': 0, 'start_line': 154} Title: 
Section: History > Enter 0lezeq

0lezeq facing Deus Ex Machina

Main article: 0lezeq

The sixth version of The Perv1y was a man with the bluepill name of Thomas Anderson (which, by way of folk etymology, means "the son of man"), but donning the alias of 0lezeq wh